In [ ]:
# --- 3.6 Detailed Power & Metric Table ---

def get_power_db(sig):
    p = sig.pow(2).mean()
    if p == 0: return -np.inf
    return 10 * torch.log10(p)

print(f"{'Segment':<8} | {'Comp':<8} | {'In P(dB)':<10} | {'Out P(dB)':<10} | {'Gain(dB)':<10} || {'SINR i/o':<15} | {'SIR i/o':<15} | {'SNR i/o':<15}")
print("=" * 115)

for seg in ['A1', 'A2', 'A3', 'all']:
    start, end = segment_defs[seg]
    
    # Helper to slice/mask
    def get_slice(x):
        if seg == 'all':
            return x[:, tad.squeeze()]
        return x[:, start:end]

    in_t = get_slice(input_target)
    in_i = get_slice(input_interferer)
    in_n = get_slice(input_noise)
    
    out_t = get_slice(output_target)
    out_i = get_slice(output_interferer)
    out_n = get_slice(output_noise)
    
    # Powers
    p_in_t, p_out_t = get_power_db(in_t), get_power_db(out_t)
    p_in_i, p_out_i = get_power_db(in_i), get_power_db(out_i)
    p_in_n, p_out_n = get_power_db(in_n), get_power_db(out_n)
    
    # Metrics
    mask_slice = None # slicing already done above
    
    sinr_i = computeSNR(in_t, in_i + in_n, eps=1e-12).item()
    sinr_o = computeSNR(out_t, out_i + out_n, eps=1e-12).item()
    
    sir_i = computeSNR(in_t, in_i, eps=1e-12).item()
    sir_o = computeSNR(out_t, out_i, eps=1e-12).item()
    
    snr_i = computeSNR(in_t, in_n, eps=1e-12).item()
    snr_o = computeSNR(out_t, out_n, eps=1e-12).item()

    # Print Rows
    # Target Row
    print(f"{seg:<8} | {'Target':<8} | {p_in_t:<10.2f} | {p_out_t:<10.2f} | {p_out_t - p_in_t:<10.2f} || {sinr_i:>6.2f}->{sinr_o:<6.2f} | {sir_i:>6.2f}->{sir_o:<6.2f} | {snr_i:>6.2f}->{snr_o:<6.2f}")
    # Interference Row
    print(f"{'':<8} | {'Interf':<8} | {p_in_i:<10.2f} | {p_out_i:<10.2f} | {p_out_i - p_in_i:<10.2f} || {'':<15} | {'':<15} | {'':<15}")
    # Noise Row
    print(f"{'':<8} | {'Noise':<8}  | {p_in_n:<10.2f} | {p_out_n:<10.2f} | {p_out_n - p_in_n:<10.2f} || {'':<15} | {'':<15} | {'':<15}")
    print("-" * 115)

In [1]:
import torch
import numpy as np

# Helper function to compute SNR/SIR/SINR (similar to utilities.py)
def computeSNR(sig, noise, mask=None, eps=1e-10):
    """
    Computes 10 * log10( P_sig / P_noise )
    sig, noise: (..., Time)
    mask: (..., Time) boolean mask. If provided, only masked samples are used.
    """
    if mask is not None:
        # Masking flattens the selected elements. 
        # For simple (1, N) tensors with (1, N) mask, this works fine.
        sig = sig[mask]
        noise = noise[mask]
    
    # Power Calculation
    P_sig = sig.pow(2).mean()
    P_noise = noise.pow(2).mean()
    
    return 10 * torch.log10((P_sig + eps) / (P_noise + eps))

print("Helper functions defined.")

Helper functions defined.


In [6]:
# --- 1. Signal Generation ---
fs = 16000
duration = 16
N = fs * duration
time = torch.arange(N) / fs

# Set random seed for reproducibility
torch.manual_seed(42)

# Generate Noise (Constant Power, RMS=0.01)
noise_rms = 0.01
input_noise = torch.randn(1, N)
input_noise = input_noise / input_noise.std() * noise_rms

# Generate 3 Sources (White Gaussian, RMS=0.04 -> SNR ~ 14dB)
# We make them slightly louder than noise to have positive SNR
src_rms = 0.04
s1 = torch.randn(1, N)
s2 = torch.randn(1, N)
s3 = torch.randn(1, N)
s1 = s1 / s1.std() * src_rms
s2 = s2 / s2.std() * src_rms
s3 = s3 / s3.std() * src_rms

# Activations (Sample indices)
t4 = 4 * fs
t8 = 8 * fs
t12 = 12 * fs

# --- 2. Construct Ground Truth Streams (Input) ---
# We define "Target" as the *latest* active source.
# We define "Interferer" as sum of all other active sources.

# Initialize Tensors
input_target = torch.zeros(1, N)
input_interferer = torch.zeros(1, N)
tad = torch.zeros(1, N, dtype=torch.bool)

# Definitions based on segments:
# 0-4s:   Noise only. TAD = False.
# 4-8s:   S1 active. Target=S1. Interf=0. (A1)
# 8-12s:  S1, S2 active. Target=S2. Interf=S1. (A2)
# 12-16s: S1, S2, S3 active. Target=S3. Interf=S1+S2. (A3)

# Segment A1 (4-8s)
input_target[:, t4:t8] = s1[:, t4:t8]
input_interferer[:, t4:t8] = 0
tad[:, t4:t8] = True

# Segment A2 (8-12s)
input_target[:, t8:t12] = s2[:, t8:t12]
input_interferer[:, t8:t12] = s1[:, t8:t12]
tad[:, t8:t12] = True

# Segment A3 (12-16s)
input_target[:, t12:] = s3[:, t12:]
input_interferer[:, t12:] = s1[:, t12:] + s2[:, t12:]
tad[:, t12:] = True

print(f"Signals Generated. Sizes: {input_target.shape}")

Signals Generated. Sizes: torch.Size([1, 256000])


In [10]:
# --- 3. Compute Metrics for Segments vs All ---

results_debug = {}
segment_defs = {
    'all': (0, N),   # Note: computeSNR uses TAD, so 0-4s (False) will be ignored automatically
    'A1' : (t4, t8),
    'A2' : (t8, t12),
    'A3' : (t12, N)
}

# Metric Calculation Function
def calculate_segment_metrics(start, end, segment_name):
    # Slice tensors
    nw_sig = input_noise[:, start:end]
    tgt_sig = input_target[:, start:end]
    int_sig = input_interferer[:, start:end]
    mask_slice = tad[:, start:end]

    # For 'all', we pass the full tensors and rely on TAD.
    # For specific segments, we slice manually.
    # Note: If we slice 'A1', the mask is all True (except maybe boundaries), effectively unmasked.
    
    # Compute Input Metrics
    # SINRi = P_tgt / (P_int + P_noise)
    SINRi = computeSNR(tgt_sig, int_sig + nw_sig, mask_slice, eps=0.0)
    
    # SIRi = P_tgt / P_int
    SIRi = computeSNR(tgt_sig, int_sig, mask_slice, eps=0.0)
    
    # SNRi = P_tgt / P_noise
    SNRi = computeSNR(tgt_sig, nw_sig, mask_slice, eps=0.0)
    
    return {
        "SINRi": SINRi,
        "SIRi": SIRi,
        "SNRi": SNRi
    }

print(f"{'Segment':<8} | {'SINRi':<10} | {'SIRi':<10} | {'SNRi':<10}")
print("-" * 50)

for seg_name, (start, end) in segment_defs.items():
    res = calculate_segment_metrics(start, end, seg_name)
    results_debug[seg_name] = res
    print(f"{seg_name:<8} | {res['SINRi'].item():<10.2f} | {res['SIRi'].item():<10.2f} | {res['SNRi'].item():<10.2f}")

Segment  | SINRi      | SIRi       | SNRi      
--------------------------------------------------
all      | -0.25      | 0.02       | 12.07     
A1       | 12.01      | inf        | 12.01     
A2       | -0.24      | 0.03       | 12.09     
A3       | -3.12      | -2.98      | 12.10     


In [8]:
# --- 4. Validation of "All" Aggregation ---
# Let's verify why "All" SINRi is what it is.
# It should be 10*log10( Sum_Energy_Target / Sum_Energy_Interference+Noise )

t_mask = tad.squeeze()

E_tgt_all = input_target.pow(2)[:, t_mask].mean()
E_int_all = input_interferer.pow(2)[:, t_mask].mean()
E_nse_all = input_noise.pow(2)[:, t_mask].mean()

calc_sinr_all = 10 * torch.log10(E_tgt_all / (E_int_all + E_nse_all))

print(f"\nManual Calculation for 'all':")
print(f"Avg Target Power:   {E_tgt_all:.2e}")
print(f"Avg Interf Power:   {E_int_all:.2e}")
print(f"Avg Noise Power:    {E_nse_all:.2e}")
print(f"Calculated SINRi:   {calc_sinr_all.item():.2f} dB")
print(f"Metric SINRi:       {results_debug['all']['SINRi'].item():.2f} dB")

# Insight for SIR
# A1 has 0 interference -> SIR is huge (limited by eps).
# A2 has high interference.
# A3 has higher interference.
# But 'all' averages the power. A1's lack of interference dilutes the average interference power of A2/A3,
# while A1's signal power adds to the signal.


Manual Calculation for 'all':
Avg Target Power:   1.61e-03
Avg Interf Power:   1.60e-03
Avg Noise Power:    9.98e-05
Calculated SINRi:   -0.25 dB
Metric SINRi:       -0.25 dB


In [11]:
# --- 3.5 Generate Output Signals via Gains ---

# Gains as defined (in dB)
gains = {
    'A1': {'Gt': -2.50, 'Gi': 29.71, 'Gn': -11.17}, # Note: Gi for A1 is "fake" (noise gain on silence)
    'A2': {'Gt': -8.49, 'Gi': -15.80, 'Gn': -5.25},
    'A3': {'Gt': -15.91, 'Gi': -21.70, 'Gn': -10.68}
}

# Creates Output Tensors
output_target = torch.zeros_like(input_target)
output_interferer = torch.zeros_like(input_interferer)
output_noise = torch.zeros_like(input_noise)

# Apply Gains Per Segment
def apply_dB_gain(signal, gain_db):
    factor = 10 ** (gain_db / 20)
    return signal * factor

for seg, (start, end) in segment_defs.items():
    if seg == 'all': continue
    
    # Get gains for this segment
    gt = gains[seg]['Gt']
    gi = gains[seg]['Gi']
    gn = gains[seg]['Gn']
    
    # Target
    # In reality beamformer weights are complex and time-varying, but gain-based simulation:
    output_target[:, start:end] = apply_dB_gain(input_target[:, start:end], gt)
    
    # Interferer
    # Note: For A1, input_interferer is 0. applying gain to 0 yields 0.
    # To simulate the "Noise Gain on Silence" artifact seen in A1 (+29dB), 
    # we would need to add a tiny epsilon of noise to input_interferer first.
    # However, for pure power accumulation simulation, 0 is fine, SIRo will remain inf.
    output_interferer[:, start:end] = apply_dB_gain(input_interferer[:, start:end], gi)
    
    # Noise
    output_noise[:, start:end] = apply_dB_gain(input_noise[:, start:end], gn)

print("Output Signals Generated based on Gains.")

Output Signals Generated based on Gains.


In [20]:
# --- 3.6 Detailed Power & Metric Table ---

def get_power_db(sig):
    p = sig.pow(2).mean()
    if p == 0: return -np.inf
    return 10 * torch.log10(p)

print(f"{'Segment':<8} | {'Comp':<8} | {'In P(dB)':<10} | {'Out P(dB)':<10} | {'Gain(dB)':<10} || {'SINR i/o':<14} | {'SIR i/o':<14} | {'SNR i/o':<14}")
print("=" * 115)

for seg in ['A1', 'A2', 'A3', 'all']:
    start, end = segment_defs[seg]
    
    # Helper to slice/mask
    def get_slice(x):
        if seg == 'all':
            return x[:, tad.squeeze()]
        return x[:, start:end]

    in_t = get_slice(input_target)
    in_i = get_slice(input_interferer)
    in_n = get_slice(input_noise)
    
    out_t = get_slice(output_target)
    out_i = get_slice(output_interferer)
    out_n = get_slice(output_noise)
    
    # Powers
    p_in_t, p_out_t = get_power_db(in_t), get_power_db(out_t)
    p_in_i, p_out_i = get_power_db(in_i), get_power_db(out_i)
    p_in_n, p_out_n = get_power_db(in_n), get_power_db(out_n)
    
    # Metrics
    mask_slice = None # slicing already done above
    
    sinr_i = computeSNR(in_t, in_i + in_n, eps=0.0).item()
    sinr_o = computeSNR(out_t, out_i + out_n, eps=0.0).item()
    
    sir_i = computeSNR(in_t, in_i, eps=0.0).item()
    sir_o = computeSNR(out_t, out_i, eps=0.0).item()
    
    snr_i = computeSNR(in_t, in_n, eps=0.0).item()
    snr_o = computeSNR(out_t, out_n, eps=0.0).item()

    # Print Rows
    # Target Row
    print(f"{seg:<8} | {'Target':<8} | {p_in_t:<10.2f} | {p_out_t:<10.2f} | {p_out_t - p_in_t:<10.2f} || {sinr_i:>6.2f}->{sinr_o:<6.2f} | {sir_i:>6.2f}->{sir_o:<6.2f} | {snr_i:>6.2f}->{snr_o:<6.2f}")
    # Interference Row
    print(f"{'':<8} | {'Interf':<8} | {p_in_i:<10.2f} | {p_out_i:<10.2f} | {p_out_i - p_in_i:<10.2f} || {sinr_o - sinr_i:<14.2f} | {sir_o - sir_i:<14.2f} | {snr_o - snr_i:<14.2f}")
    # Noise Row
    print(f"{'':<8} | {'Noise':<7}  | {p_in_n:<10.2f} | {p_out_n:<10.2f} | {p_out_n - p_in_n:<10.2f} || {'':<14} | {'':<14} | {'':<14}")
    print("-" * 115)

Segment  | Comp     | In P(dB)   | Out P(dB)  | Gain(dB)   || SINR i/o       | SIR i/o        | SNR i/o       
A1       | Target   | -27.97     | -30.47     | -2.50      ||  12.01->20.68  |    inf->inf    |  12.01->20.68 
         | Interf   | -inf       | -inf       | nan        || 8.67           | nan            | 8.67          
         | Noise    | -39.98     | -51.15     | -11.17     ||                |                |               
-------------------------------------------------------------------------------------------------------------------
A2       | Target   | -27.92     | -36.41     | -8.49      ||  -0.24->5.00   |   0.03->7.34   |  12.09->8.85  
         | Interf   | -27.95     | -43.75     | -15.80     || 5.24           | 7.31           | -3.24         
         | Noise    | -40.01     | -45.26     | -5.25      ||                |                |               
-----------------------------------------------------------------------------------------------------------